<a href="https://colab.research.google.com/github/chervov/ProteinModelling1/blob/main/colab/tyk2_ddg_clash_diagnosis.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TYK2 ddG: are the extreme values real?

The dual-state ddG run in `data/ddg/` produces values down to **-3000**, far outside any physical folding free energy. This notebook asks whether those numbers are **steric clash artefacts** rather than energies, and whether the pipeline's protocol is to blame.

| protocol | what it does |
|---|---|
| **A** | the pipeline: `mutate_residue(pack_radius=8)`, side chains only |
| **fa_rep** | how much of the ddG is the Lennard-Jones repulsive term |
| **B** | A plus backbone + side-chain minimisation of the 8 A shell |
| **ddGMover** | Rosetta's own `ddg_monomer` (Kellogg 2011): repulsive ramp, constrained minimisation, several decoys |

Sign convention follows the paper: **ddG = E(WT) - E(mut)**, so *negative = destabilising*. Rosetta's own convention is the opposite, which is why the ddGMover numbers are positive.

**Runtime:** a few minutes. The relaxed structure is downloaded rather than recomputed; relaxing it from scratch would take ~35 min.

Companion write-up: `conclusions_ddG_difference_feature.md`, sections 8-10.

## 1. Install PyRosetta

Downloads ~1.5 GB, so this cell takes a few minutes. PyRosetta is free for academic use; commercial use requires a licence from UW.

In [ ]:
import sys, subprocess

try:
    import pyrosetta
    print("PyRosetta already present")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyrosetta-installer"], check=True)
    import pyrosetta_installer
    pyrosetta_installer.install_pyrosetta()
    print("installed - if the import below fails, restart the runtime and re-run")

## 2. Structure

The same AlphaFold-3 closed-state model the production run used, already relaxed with FastRelax under coordinate constraints and published in the repo.

Watch the **total REU**: that single number is the yardstick for judging every ddG below.

In [ ]:
import os, time, json, gzip, urllib.request
import pyrosetta
from pyrosetta import pose_from_pdb, get_fa_scorefxn
from pyrosetta.toolbox import mutate_residue
from pyrosetta.rosetta.core.scoring import ScoreType
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.protocols.minimization_packing import MinMover

RAW = "https://raw.githubusercontent.com/chervov/ProteinModelling1/main/"
WORK = "/content/ddg"
SHELL = 8.0
TEST_POSITIONS = [746, 964, 801, 667, 775]   # first three are glycines with the largest stored ddG
TEST_ALTS = ["W", "F", "A", "G"]
os.makedirs(WORK, exist_ok=True)

pyrosetta.init("-mute all -ex1 -ex2aro -ignore_unrecognized_res "
               "-ignore_zero_occupancy false -detect_disulf false")
scorefxn = get_fa_scorefxn()

pdb = f"{WORK}/TYK2_relaxed_closed.pdb"
if not os.path.exists(pdb):
    with urllib.request.urlopen(RAW + "data/ddg/TYK2_relaxed_closed.pdb.gz", timeout=300) as r:
        open(pdb, "wb").write(gzip.decompress(r.read()))
pose = pose_from_pdb(pdb)
total_wt = scorefxn(pose)
print(f"{pose.total_residue()} residues, {total_wt:.0f} REU total "
      f"({total_wt / pose.total_residue():.2f} REU/residue)")

## 3. Protocols A and B

For each substitution: the pipeline's value, the same with a mobile backbone, and how much of it is pure steric overlap.

In [ ]:
def weighted_terms(p):
    """Weighted score terms of a pose (fa_rep = Lennard-Jones repulsion)."""
    scorefxn(p)
    totals = p.energies().total_energies()
    return {st.name: totals[st] * scorefxn.get_weight(st)
            for st in scorefxn.get_nonzero_weighted_scoretypes()}

def minimise_shell(p, position, radius=SHELL):
    """Protocol B: let backbone AND side chains near the mutation move."""
    mm = MoveMap()
    centre = p.residue(position).xyz("CA")
    for i in range(1, p.total_residue() + 1):
        if p.residue(i).has("CA") and (p.residue(i).xyz("CA") - centre).norm() <= radius:
            mm.set_chi(i, True); mm.set_bb(i, True)
    MinMover(mm, scorefxn, "lbfgs_armijo_nonmonotone", 0.01, True).apply(p)
    return p

def ddg_for(position, alt):
    """ddG = E(WT) - E(mutant); negative = destabilising. The WT is repacked identically."""
    native = pose.residue(position).name1()
    if native == alt:
        return None
    wt = pose.clone(); mutate_residue(wt, position, native, pack_radius=SHELL)
    mut = pose.clone(); mutate_residue(mut, position, alt, pack_radius=SHELL)
    a = scorefxn(wt) - scorefxn(mut)
    d_rep = weighted_terms(mut).get("fa_rep", 0.0) - weighted_terms(wt).get("fa_rep", 0.0)
    b = scorefxn(minimise_shell(wt.clone(), position)) - scorefxn(minimise_shell(mut.clone(), position))
    return {"variant": f"{native}{position}{alt}", "A_repack_only": round(a, 1),
            "B_flexible_bb": round(b, 1),
            "fa_rep_share": round(-d_rep / a, 2) if abs(a) > 1e-9 else None}

print(f"{'variant':8s} {'A (repack only)':>16s} {'B (flexible bb)':>16s} {'steric':>8s}")
print('-' * 52)
results = []
for _pos in TEST_POSITIONS:
    for _alt in TEST_ALTS:
        r = ddg_for(_pos, _alt)
        if r:
            results.append(r)
            s = f"{r['fa_rep_share']:.0%}" if r['fa_rep_share'] is not None else '-'
            print(f"{r['variant']:8s} {r['A_repack_only']:16.1f} {r['B_flexible_bb']:16.1f} {s:>8s}")
print('-' * 52)
print(f"{len(results)} variants. More negative = more destabilising.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ddg_table = pd.DataFrame(results).sort_values("A_repack_only")
ddg_table["shrink_factor"] = (ddg_table.A_repack_only / ddg_table.B_flexible_bb).round(1)
display(ddg_table)

fig, ax = plt.subplots(figsize=(7, 6))
y = range(len(ddg_table))
ax.barh([i + 0.2 for i in y], ddg_table.A_repack_only, height=0.4, label="A: repack only")
ax.barh([i - 0.2 for i in y], ddg_table.B_flexible_bb, height=0.4, label="B: flexible backbone")
ax.set_yticks(list(y)); ax.set_yticklabels(ddg_table.variant)
ax.set_xlabel("ddG (REU, negative = destabilising)"); ax.legend(); ax.set_title("Repack-only vs flexible backbone")
plt.tight_layout(); plt.show()

## 4. Cross-check: Rosetta's own ddG application

The obvious suspicion is that the repack-only protocol is at fault. `ddGMover` **is** `ddg_monomer` (Kellogg et al. 2011) and does everything the pipeline lacks: ramps the repulsive term, minimises under coordinate constraints, takes the best of several decoys.

The table shows the stored run. `run_ddg_monomer()` recomputes any variant, at roughly **6 minutes each**.

In [ ]:
def run_ddg_monomer(position, alt, iterations=3):
    """Rosetta's own ddG protocol. Returns (REU, kcal/mol, seconds). Positive = destabilising."""
    from pyrosetta.rosetta.protocols.ddg import ddGMover
    from pyrosetta.rosetta.utility import vector1_core_chemical_AA
    from pyrosetta.rosetta.core.chemical import aa_from_oneletter_code
    muts = vector1_core_chemical_AA(pose.total_residue())
    for i in range(1, pose.total_residue() + 1):
        muts[i] = pose.residue(i).aa()
    muts[position] = aa_from_oneletter_code(alt)
    mover = ddGMover(scorefxn, scorefxn, muts)
    mover.set_min_cst(True); mover.set_min(True); mover.set_mean(False)
    mover.restrict_to_nbrs(True); mover.num_iterations(iterations)
    work = pose.clone()
    t0 = time.time(); mover.apply(work)
    reu = mover.ddG()
    return reu, reu / 2.94, time.time() - t0   # 2.94 = documented REU -> kcal/mol factor

kellogg_table = pd.DataFrame(
    json.load(urllib.request.urlopen(RAW + "data/ddg/kellogg_ddg_monomer.json", timeout=120)))
display(kellogg_table)

# Uncomment to recompute one yourself (~6 min):
# print(run_ddg_monomer(667, 'F'))

## What this shows

**The extreme values are not energies.** The whole folded protein scores about **-2689 REU**. A ddG of -3000 puts the mutant at **+311 REU** - a positive total energy, which no folded structure has. Their `fa_rep` share is ~100%: the ddG *is* the repulsive term and nothing else.

**But the protocol is not the culprit.** Rosetta's own `ddg_monomer` returns **+2832 REU for G746W**, the same magnitude as the pipeline's -2989, while giving textbook values for ordinary mutations (**H667F 2.08 kcal/mol**, **R775A 0.62 kcal/mol**). Two independently written protocols agree, so there is no implementation defect.

**What is really going on.** Every extreme position is a buried **glycine**. No standard ddG method rearranges the backbone enough to host a tryptophan in a glycine-sized pocket, so they all report the same saturated *"does not fit"* flag, scaled by how deep the overlap goes. G->A at the same positions is 10-50x milder.

**What to do with the values.** Rank them; never quote them as energies. That is what the pipeline README already said: *"only the ordering is meaningful"*, and it is why every result in `conclusions_ddG_difference_feature.md` is rank-based.